# 13 · Real-MOSEI cache for fast uncertainty experiments (CPU, no training)

Dumps everything the small experiments need, so each one runs in seconds instead of re-running the experts:
for three trained models (frozen V-JEPA PoE, frozen variational-target PoE, fine-tuned V-JEPA PoE) and every
test condition, the label belief of **every availability pattern** (all three modalities, each pair, each single
modality), plus the pooled input statistics (model-independent).

* train / valid: clean + self-generated corruptions (white noise λ = 1 and random swap on each modality), for
  techniques that learn from synthetic corruptions;
* test: clean, the same corruptions, and held-out ones (noise λ = 0.5, half-clip noise, gain fault, dead sensor,
  time shuffle, nearest-neighbour swap, two noisy modalities).

Attach the outputs of 01, 03 and 07. Output: `uq_cache.npz` (float16).

In [ ]:
# Setup: make the `fusionlab` library importable. On Kaggle, always use a FRESH clone of the latest code (kept in /tmp,
# so it is not saved into this notebook's output); attached copies are used only when there is no Internet.
import glob, os, subprocess, sys

REPO_URL = 'https://github.com/fasihh/bigthing'      # the JF/ folder holds this project
BRANCH = 'jf-experiments'                           # used only until JF/ is merged into main


def fresh_clone(dest='/tmp/bigthing'):
    if not os.path.isdir(os.path.join(dest, '.git')):
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], capture_output=True, text=True)
        if r.returncode:
            return None
    if not os.path.isdir(os.path.join(dest, 'JF')):  # JF not merged into main yet: use its branch
        subprocess.run(['git', '-C', dest, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', dest, 'checkout', '-q', 'FETCH_HEAD'], check=True)
    return os.path.join(dest, 'JF')


def find_jf():
    here = os.path.abspath('.')
    if not here.startswith('/kaggle'):                # running inside a local checkout of the repo
        for c in (os.path.dirname(here), here, os.path.join(here, 'JF')):
            if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
                return c
    jf = fresh_clone()
    if jf:
        return jf
    for c in glob.glob('/kaggle/input/**/JF', recursive=True):   # offline fallback (may be older code)
        if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
            print('WARNING: no Internet, using an attached copy of the code:', c)
            return c
    raise RuntimeError('Cannot find the fusionlab code: turn on Internet in the notebook settings.')


JF = find_jf()
sys.path.insert(0, os.path.join(JF, 'src'))
CONFIGS = os.path.join(JF, 'configs')
commit = subprocess.run(['git', '-C', JF, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('using fusionlab from', JF, '| commit', commit or 'n/a')

In [ ]:
import time
import numpy as np, torch
from fusionlab import BASE, config, data, gate, pretrain, runner
from fusionlab.runtime import RT

cfg = config.load_config(os.path.join(CONFIGS, '12_mosei_gate.yaml'))
data.load_mosei(cfg)
OUT = cfg['out_dir']
MODELS = [('vjepa', 'frozen'), ('vjepa_vt', 'frozen'), ('vjepa', 'finetuned')]
SEED = 0

tr_mask = {m: RT.DATA['train'][m + 'm'] for m in BASE}
feat_mean = {m: RT.DATA['train'][m][tr_mask[m]].mean(0).to(RT.DEV) for m in BASE}
self_gen = [dict(name='clean', kind='clean')] + \
    [dict(name=f'noise[{m}]', kind='noise', m=m, lam=1.0) for m in BASE] + \
    [dict(name=f'swap[{m}]', kind='swap', m=m) for m in BASE]
test_conds = [c for c in gate.gate_conditions(include_heldout=True)]

In [ ]:
store = {}
perms = {s: dict(swap=torch.randperm(RT.N_SPLIT[s], generator=torch.Generator().manual_seed(123 + 7))) for s in RT.N_SPLIT}
for obj, rg in MODELS:
    if not cfg['quick'] and not pretrain.find_ckpt(f'v2_model_main_{obj}_{rg}_poe_s{SEED}.pt'):
        print('skip', obj, rg); continue
    t0 = time.time()
    experts, _, marg = pretrain.pretrain_or_load(obj, BASE, 'main', SEED, cfg)
    model, _ = runner.build_model(experts, obj, rg, 'poe', BASE, cfg, SEED, marg, 'main')
    tag = f'{obj}_{rg}'
    for split, conds in (('train', self_gen), ('valid', self_gen), ('test', test_conds)):
        clean = None
        for cd in conds:
            if cd['kind'] == 'swapnn' and 'swapnn' not in perms[split]:
                perms[split]['swapnn'] = {m: gate.nn_perm(clean['F'], m) for m in BASE}
            r = gate.run_condition(model, split, cd, perms[split], feat_mean, singles=True)
            if cd['kind'] == 'clean':
                clean = r
            key = f'{split}/{cd["name"]}'
            pats = list(r['pred'])
            store[f'{key}/{tag}/mu'] = np.stack([r['pred'][p][0] for p in pats], 1).astype(np.float32)
            store[f'{key}/{tag}/var'] = np.stack([r['pred'][p][1] for p in pats], 1).astype(np.float32)
            if f'{key}/F' not in store:
                store[f'{key}/F'] = np.concatenate([r['F'][m] for m in BASE], 1).astype(np.float16)
            store[f'{split}/y'] = r['y'].astype(np.float32)
    store[f'patterns'] = np.array([''.join(m for m, a in zip(BASE, p) if a) for p in pats])
    print(tag, f'{time.time() - t0:.0f}s', flush=True)

store['fdims'] = np.array([RT.DIMS[m] * 2 for m in BASE])
np.savez_compressed(os.path.join(OUT, 'uq_cache.npz'), **store)
print('saved', os.path.getsize(os.path.join(OUT, 'uq_cache.npz')) / 1e6, 'MB,', len(store), 'arrays')